In [1]:
import re
import time
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

print("TensorFlow version:", tf.__version__)


TensorFlow version: 2.21.0


In [2]:
df = pd.read_csv(r"C:\Users\Bhargav\Downloads\DATASETS\CEAS_08.csv")
print("Shape:", df.shape)
df.head(3)

Shape: (39154, 7)


,sender,receiver,date,subject,body,label,urls
0,Young Esposito <Young@iworld.de>,user4@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 16:31:02 -0700",Never agree to be a loser,"Buck up, your troubles caused by small dimensi...",1,1
1,Mok <ipline's1983@icable.ph>,user2.2@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 18:31:03 -0500",Befriend Jenna Jameson,\nUpgrade your sex and pleasures with these te...,1,1
2,Daily Top 10 <Karmandeep-opengevl@universalnet...,user2.9@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 20:28:00 -1200",CNN.com Daily Top 10,>+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+...,1,1


In [3]:
df.duplicated().sum()

np.int64(0)

In [4]:
df.drop_duplicates(inplace  = True)

In [5]:
df.isna().sum()

sender        0
receiver    462
date          0
subject      28
body          0
label         0
urls          0
dtype: int64

In [6]:
df = df.fillna("Unknown")

In [7]:
df = df[
[
    "subject",
    "body",
    "label",
    "urls"
]
]

In [8]:
df["email_text"] = df["subject"] + " " + df["body"]

In [9]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"<.*?>", " ", text)        # remove HTML tags e.g. <br />
    text = re.sub(r"[^a-z\s]", " ", text)     # keep only letters
    text = re.sub(r"\s+", " ", text).strip()  # collapse whitespace
    return text

In [10]:
df["email_text"] = df["email_text"].apply(clean_text)

In [11]:
X = df["email_text"]
y = df["label"]

In [12]:
X_train_text,X_test_text,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

## 6. Prepare Sequences for the Neural Networks

### Tokenization
Convert each cleaned review into a sequence of integers, where each integer represents a word's rank in the vocabulary (fit **only** on the training set).

### Padding
Neural networks need fixed-length input. We pad/truncate every sequence to `max_len=200`.


In [14]:
tokenizer = Tokenizer(num_words=10000, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_text)

X_train_seq = tokenizer.texts_to_sequences(X_train_text)
X_test_seq = tokenizer.texts_to_sequences(X_test_text)

X_train_pad = pad_sequences(X_train_seq, maxlen=200, padding="post", truncating="post")
X_test_pad = pad_sequences(X_test_seq, maxlen=200, padding="post", truncating="post")

y_train_arr = y_train.values
y_test_arr = y_test.values


In [18]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow as tf


rnn_model = Sequential([

    Embedding(
        input_dim=10000,
        output_dim=64,
        mask_zero=True
    ),

    SimpleRNN(64),

    Dropout(0.3),

    Dense(64, activation="relu"),

    Dense(1, activation="sigmoid")

])

rnn_model.compile(

    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),

    loss = "binary_crossentropy",

    metrics=["accuracy"]

)

early_stop = EarlyStopping(

    monitor="val_loss",

    patience=4,

    restore_best_weights=True

)

history_rnn = rnn_model.fit(

    X_train_pad,

    y_train,

    validation_split=0.1,

    epochs=15,

    batch_size=256,

    callbacks=[early_stop]

)

Epoch 1/15
111/111 ━━━━━━━━━━━━━━━━━━━━ 16s 125ms/step - accuracy: 0.8896 - loss: 0.2890 - val_accuracy: 0.9885 - val_loss: 0.0504
Epoch 2/15
111/111 ━━━━━━━━━━━━━━━━━━━━ 14s 129ms/step - accuracy: 0.9918 - loss: 0.0326 - val_accuracy: 0.9943 - val_loss: 0.0262
Epoch 3/15
111/111 ━━━━━━━━━━━━━━━━━━━━ 12s 112ms/step - accuracy: 0.9961 - loss: 0.0157 - val_accuracy: 0.9939 - val_loss: 0.0250
Epoch 4/15
111/111 ━━━━━━━━━━━━━━━━━━━━ 13s 115ms/step - accuracy: 0.9981 - loss: 0.0081 - val_accuracy: 0.9965 - val_loss: 0.0190
Epoch 5/15
111/111 ━━━━━━━━━━━━━━━━━━━━ 13s 113ms/step - accuracy: 0.9995 - loss: 0.0029 - val_accuracy: 0.9962 - val_loss: 0.0171
Epoch 6/15
111/111 ━━━━━━━━━━━━━━━━━━━━ 12s 105ms/step - accuracy: 0.9984 - loss: 0.0058 - val_accuracy: 0.9939 - val_loss: 0.0278
Epoch 7/15
111/111 ━━━━━━━━━━━━━━━━━━━━ 11s 103ms/step - accuracy: 0.9988 - loss: 0.0041 - val_accuracy: 0.9933 - val_loss: 0.0372
Epoch 8/15
111/111 ━━━━━━━━━━━━━━━━━━━━ 11s 99ms/step - accuracy: 0.9976 - loss: 0.

In [22]:
y_pred_rnn = (rnn_model.predict(X_test_pad) > 0.5).astype(int)

print(classification_report(y_test_arr, y_pred_rnn))


245/245 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step
              precision    recall  f1-score   support

           0       0.99      1.00      1.00      3490
           1       1.00      0.99      1.00      4341

    accuracy                           1.00      7831
   macro avg       1.00      1.00      1.00      7831
weighted avg       1.00      1.00      1.00      7831



In [23]:
accuracy_score(y_test_arr, y_pred_rnn)

0.9961690716383603